In [3]:
# Setup output directory for generated visualizations
import os
output_dir = "output_visualizations"
os.makedirs(output_dir, exist_ok=True)

# Criterion Channel Advertising Experiment Analysis

This notebook presents a structured analysis of campaign effectiveness, conversion lift, exposure effects, and channel-level ROI using the Criterion Channel experiment data.

## 1) Setup and Data Load
Load the experiment dataset and configure display options for inspection.

In [4]:
import numpy as np
import pandas as pd
from scipy.stats import norm, ttest_ind
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", None)
df = pd.read_csv("data/criterion.csv")
df.head()

,id,subscription,imp_facebook,imp_twitter,imp_instagram,imp_youtube,imp_letterboxd,treatment
0,917978073,1,0,0,0,0,0,1
1,630918244,1,0,0,0,0,0,1
2,663985446,1,0,0,0,0,0,1
3,572576568,1,0,0,0,0,0,1
4,673572395,1,0,23,0,0,0,1


In [5]:
# how many in control vs treatment
df["treatment"].value_counts(normalize=True)

treatment
1    0.700047
0    0.299953
Name: proportion, dtype: float64

## 2) Data Overview
Validate dataset dimensions and core schema before modeling.

In [6]:
print("Shape:", df.shape)
print("Columns:", list(df.columns))

Shape: (1200093, 8)
Columns: ['id', 'subscription', 'imp_facebook', 'imp_twitter', 'imp_instagram', 'imp_youtube', 'imp_letterboxd', 'treatment']


## 3) Exposure Feature Engineering
Create total impressions per user by summing channel-level impressions.

In [7]:
impression_columns = [col for col in df.columns if "imp" in col]
df["total_impressions"] = df[impression_columns].sum(axis=1)
total_impressions = int(df["total_impressions"].sum())
print("Total impressions in experiment:", total_impressions)

Total impressions in experiment: 3924541


## 4) Randomization Diagnostic
Check whether treatment and control have comparable exposure distributions.

In [8]:
control_imp = df.loc[df["treatment"] == 0, "total_impressions"]
treat_imp = df.loc[df["treatment"] == 1, "total_impressions"]
imp_ttest = ttest_ind(control_imp, treat_imp)

z = norm.ppf(0.975)
control_mean = control_imp.mean()
treat_mean = treat_imp.mean()
control_se = control_imp.std(ddof=1) / np.sqrt(control_imp.shape[0])
treat_se = treat_imp.std(ddof=1) / np.sqrt(treat_imp.shape[0])

mean_diff = treat_mean - control_mean
diff_se = np.sqrt((treat_imp.var(ddof=1) / treat_imp.shape[0]) + (control_imp.var(ddof=1) / control_imp.shape[0]))

randomization_summary = pd.DataFrame({
    "group": ["control", "treatment"],
    "mean_total_impressions": [control_mean, treat_mean],
    "ci95_low": [control_mean - z * control_se, treat_mean - z * treat_se],
    "ci95_high": [control_mean + z * control_se, treat_mean + z * treat_se],
})

randomization_diff_summary = pd.DataFrame({
    "metric": ["treatment_minus_control_mean_impressions"],
    "estimate": [mean_diff],
    "ci95_low": [mean_diff - z * diff_se],
    "ci95_high": [mean_diff + z * diff_se],
    "p_value": [imp_ttest[1]],
})

channel_balance = []
for col in impression_columns:
    c = df.loc[df["treatment"] == 0, col]
    t = df.loc[df["treatment"] == 1, col]
    p = ttest_ind(c, t)[1]
    channel_balance.append({
        "channel": col,
        "p_value": p,
        "balanced_at_5pct": p >= 0.05
    })

display(randomization_summary)
display(randomization_diff_summary)
display(pd.DataFrame(channel_balance))

,group,mean_total_impressions,ci95_low,ci95_high
0,control,3.580166,3.530320,3.630013
1,treatment,3.137384,3.111227,3.163540


,metric,estimate,ci95_low,ci95_high,p_value
0,treatment_minus_control_mean_impressions,-0.442783,-0.499075,-0.38649,1.701588e-63


,channel,p_value,balanced_at_5pct
0,imp_facebook,5.523948e-204,False
1,imp_twitter,1.567417e-01,True
2,imp_instagram,0.000000e+00,False
3,imp_youtube,2.278152e-20,False
4,imp_letterboxd,3.436258e-163,False


## 5) Treatment Effect on Conversion
Estimate average conversion impact of treatment versus control.

In [9]:
conv_rates = df.groupby("treatment")["subscription"].mean()
control_conv = conv_rates.loc[0]
treat_conv = conv_rates.loc[1]
abs_lift_pp = (treat_conv - control_conv) * 100
rel_lift_pct = ((treat_conv - control_conv) / control_conv) * 100

control_sub = df.loc[df["treatment"] == 0, "subscription"]
treat_sub = df.loc[df["treatment"] == 1, "subscription"]
conv_p = ttest_ind(control_sub, treat_sub)[1]

n_control = control_sub.shape[0]
n_treat = treat_sub.shape[0]
z = norm.ppf(0.975)

se_control = np.sqrt((control_conv * (1 - control_conv)) / n_control)
se_treat = np.sqrt((treat_conv * (1 - treat_conv)) / n_treat)
se_diff = np.sqrt((treat_conv * (1 - treat_conv)) / n_treat + (control_conv * (1 - control_conv)) / n_control)

effect_summary = pd.DataFrame({
    "metric": [
        "control_conversion_rate",
        "treatment_conversion_rate",
        "absolute_lift_pp",
        "relative_lift_pct",
        "p_value"
    ],
    "value": [control_conv, treat_conv, abs_lift_pp, rel_lift_pct, conv_p],
    "ci95_low": [
        control_conv - z * se_control,
        treat_conv - z * se_treat,
        abs_lift_pp - z * se_diff * 100,
        np.nan,
        np.nan,
    ],
    "ci95_high": [
        control_conv + z * se_control,
        treat_conv + z * se_treat,
        abs_lift_pp + z * se_diff * 100,
        np.nan,
        np.nan,
    ],
})

display(effect_summary)

,metric,value,ci95_low,ci95_high
0,control_conversion_rate,0.002992,0.002813,0.003170
1,treatment_conversion_rate,0.013409,0.013163,0.013655
2,absolute_lift_pp,1.041686,1.011301,1.072071
3,relative_lift_pct,348.167800,NaN,NaN
4,p_value,0.000000,NaN,NaN


## 6) Frequency Response Analysis
Measure how incremental conversion lift changes across impression-volume bands.

In [10]:
groups = ["0-40", "41-80", "81-120", "121-160", "161-200", "201-240", "241-280", "281-320", "320+"]
bins = [0, 40, 80, 120, 160, 200, 240, 280, 320, df["total_impressions"].max()]
df["impression_group"] = pd.cut(df["total_impressions"], bins=bins, labels=groups, include_lowest=True)

z = norm.ppf(0.975)
results = []
for group in groups:
    temp = df[df["impression_group"] == group]

    t = temp.loc[temp["treatment"] == 1, "subscription"]
    c = temp.loc[temp["treatment"] == 0, "subscription"]

    n_t = t.shape[0]
    n_c = c.shape[0]
    t_mean = t.mean()
    c_mean = c.mean()

    test_rate = t_mean * 100
    control_rate = c_mean * 100
    p_value = ttest_ind(t, c)[1] if n_t > 1 and n_c > 1 else np.nan
    abs_lift = (t_mean - c_mean) * 100

    if pd.notna(p_value) and p_value < 0.05:
        direction = "test_higher" if t_mean >= c_mean else "test_lower"
        significance = f"significant_{direction}"
    else:
        significance = "not_significant"

    test_se = np.sqrt((t_mean * (1 - t_mean)) / n_t) if n_t > 0 else np.nan
    control_se = np.sqrt((c_mean * (1 - c_mean)) / n_c) if n_c > 0 else np.nan
    diff_se = np.sqrt((t_mean * (1 - t_mean)) / n_t + (c_mean * (1 - c_mean)) / n_c) if n_t > 0 and n_c > 0 else np.nan

    rel_lift = abs_lift / control_rate if pd.notna(control_rate) and control_rate != 0 else pd.NA

    results.append({
        "Group": group,
        "Test Conversion Rate (%)": test_rate,
        "Test CI95 Low (%)": test_rate - z * test_se * 100 if pd.notna(test_se) else np.nan,
        "Test CI95 High (%)": test_rate + z * test_se * 100 if pd.notna(test_se) else np.nan,
        "Control Conversion Rate (%)": control_rate,
        "Control CI95 Low (%)": control_rate - z * control_se * 100 if pd.notna(control_se) else np.nan,
        "Control CI95 High (%)": control_rate + z * control_se * 100 if pd.notna(control_se) else np.nan,
        "P-value": p_value,
        "Significance": significance,
        "Absolute Lift (%)": abs_lift,
        "Lift CI95 Low (%)": abs_lift - z * diff_se * 100 if pd.notna(diff_se) else np.nan,
        "Lift CI95 High (%)": abs_lift + z * diff_se * 100 if pd.notna(diff_se) else np.nan,
        "Relative Lift (%)": rel_lift,
        "Test Impressions": n_t,
        "Control Impressions": n_c
    })

results_df = pd.DataFrame(results)
display(results_df)

/opt/anaconda3/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


,Group,Test Conversion Rate (%),Test CI95 Low (%),Test CI95 High (%),Control Conversion Rate (%),Control CI95 Low (%),Control CI95 High (%),P-value,Significance,Absolute Lift (%),Lift CI95 Low (%),Lift CI95 High (%),Relative Lift (%),Test Impressions,Control Impressions
0,0-40,1.286102,1.261855,1.310350,0.288651,0.271006,0.306296,0.000000e+00,significant_test_higher,0.997451,0.967463,1.027440,3.45556,829483,355100
1,41-80,6.025621,5.439088,6.612153,1.097179,0.693021,1.501336,9.181202e-24,significant_test_higher,4.928442,4.216147,5.640737,4.491923,6323,2552
2,81-120,4.815276,3.960359,5.670193,1.384768,0.664435,2.105100,1.637699e-06,significant_test_higher,3.430508,2.312581,4.548436,2.477317,2409,1011
3,121-160,6.135770,4.436281,7.835259,1.486989,0.040641,2.933336,2.416499e-03,significant_test_higher,4.648781,2.417149,6.880414,3.126305,766,269
4,161-200,2.695035,1.499663,3.890408,0.384123,-0.049709,0.817955,2.254990e-04,significant_test_higher,2.310913,1.039250,3.582575,6.016076,705,781
5,201-240,26.760563,16.462892,37.058235,100.000000,100.000000,100.000000,6.089077e-03,significant_test_lower,-73.239437,-83.537108,-62.941765,-0.732394,71,3
6,241-280,1.315789,-0.163311,2.794890,NaN,NaN,NaN,NaN,not_significant,NaN,NaN,NaN,<NA>,228,0
7,281-320,3.030303,-1.105284,7.165890,NaN,NaN,NaN,NaN,not_significant,NaN,NaN,NaN,<NA>,66,0
8,320+,14.084507,5.993073,22.175941,0.000000,0.000000,0.000000,4.170860e-10,significant_test_higher,14.084507,5.993073,22.175941,<NA>,71,255


## 7) Interaction Model for Exposure Intensity
Fit a logistic model with an exposure-by-treatment interaction to quantify how treatment impact scales with impressions.

In [11]:
model = smf.logit("subscription ~ total_impressions + total_impressions:treatment", data=df).fit()
model.summary()

Optimization terminated successfully.
         Current function value: 0.056719
         Iterations 9


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:           subscription   No. Observations:              1200093
Model:                          Logit   Df Residuals:                  1200090
Method:                           MLE   Df Model:                            2
Date:                Mon, 16 Mar 2026   Pseudo R-squ.:                 0.01020
Time:                        15:56:42   Log-Likelihood:                -68068.
converged:                       True   LL-Null:                       -68769.
Covariance Type:            nonrobust   LLR p-value:                3.107e-305
===============================================================================================
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                      -4.6143      0.009   -491.865      0.000      -4.633      -4.596
total_impressions              -0.0068      0.002     -3.745      0.000      -0.010      -0.003
total_impressions:treatment     0.0207      0.002     11.307      0.000       0.017       0.024
===============================================================================================
"""

In [12]:
# Diminishing-returns exposure model: log(1 + impressions)

model_log = smf.logit(

    "subscription ~ treatment + np.log1p(total_impressions) + treatment:np.log1p(total_impressions)",

    data=df

).fit()



# Compare linear vs diminishing-returns fit

fit_compare = pd.DataFrame({

    "model": ["linear_impressions", "log1p_impressions"],

    "aic": [model.aic, model_log.aic],

    "bic": [model.bic, model_log.bic],

    "log_likelihood": [model.llf, model_log.llf],

})

display(fit_compare.sort_values("aic"))



# Dose-response table: predicted conversion and incremental lift by exposure level

imp_grid = np.quantile(df["total_impressions"], [0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0])

curve_df = pd.DataFrame({"total_impressions": np.unique(np.round(imp_grid, 0))})



curve_treat = curve_df.assign(treatment=1)

curve_control = curve_df.assign(treatment=0)



curve_df["pred_control"] = model_log.predict(curve_control)

curve_df["pred_treatment"] = model_log.predict(curve_treat)

curve_df["incremental_lift_pp"] = (curve_df["pred_treatment"] - curve_df["pred_control"]) * 100



display(curve_df)

model_log.summary()

Optimization terminated successfully.
         Current function value: 0.054371
         Iterations 10


,model,aic,bic,log_likelihood
1,log1p_impressions,130507.987724,130555.979362,-65249.993862
0,linear_impressions,136142.603593,136178.597322,-68068.301796


,total_impressions,pred_control,pred_treatment,incremental_lift_pp
0,0.0,0.002012,0.008256,0.624457
1,1.0,0.002661,0.011621,0.896006
2,2.0,0.003134,0.014185,1.105112
3,6.0,0.004409,0.021472,1.706353
4,397.0,0.022199,0.141033,11.883426


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:           subscription   No. Observations:              1200093
Model:                          Logit   Df Residuals:                  1200089
Method:                           MLE   Df Model:                            3
Date:                Mon, 16 Mar 2026   Pseudo R-squ.:                 0.05118
Time:                        15:56:45   Log-Likelihood:                -65250.
converged:                       True   LL-Null:                       -68769.
Covariance Type:            nonrobust   LLR p-value:                     0.000
=========================================================================================================
                                            coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------
Intercept                                -6.2068      0.043   -144.448      0.000      -6.291      -6.123
treatment                                 1.4183      0.045     31.464      0.000       1.330       1.507
np.log1p(total_impressions)               0.4045      0.024     16.557      0.000       0.357       0.452
treatment:np.log1p(total_impressions)     0.0936      0.026      3.657      0.000       0.043       0.144
=========================================================================================================
"""

## 8) Channel-Level Incremental Lift and ROI
Estimate each channel's incremental contribution by removing channel-specific treatment effects in counterfactual predictions.

In [13]:
df["imp_facebookT"] = df["imp_facebook"] * df["treatment"]
df["imp_twitterT"] = df["imp_twitter"] * df["treatment"]
df["imp_instagramT"] = df["imp_instagram"] * df["treatment"]
df["imp_letterboxdT"] = df["imp_letterboxd"] * df["treatment"]
df["imp_youtubeT"] = df["imp_youtube"] * df["treatment"]

model2 = smf.logit(
    "subscription ~ imp_facebook + imp_facebookT + imp_twitter + imp_twitterT + imp_instagram + imp_instagramT + imp_letterboxd + imp_letterboxdT + imp_youtube + imp_youtubeT",
    data=df
).fit()

dfT = df[df["treatment"] == 1].copy()
dfT["pred_all"] = model2.predict(dfT)

channels = ["facebook", "twitter", "instagram", "letterboxd", "youtube"]
rows = []
lifts = {}

rng = np.random.default_rng(42)
n_boot = 300

for ch in channels:
    temp = dfT.copy()
    temp[f"imp_{ch}T"] = 0
    pred_no_ch = model2.predict(temp)

    delta = dfT["pred_all"] - pred_no_ch
    lift = delta.sum()
    cost = dfT[f"imp_{ch}"].sum() * 0.03
    roi = (lift * 1300 - cost) / cost

    lifts[ch] = lift

    boot_lifts = []
    for _ in range(n_boot):
        idx = rng.integers(0, delta.shape[0], size=delta.shape[0])
        sampled_delta = delta.iloc[idx]
        boot_lifts.append(sampled_delta.sum())

    boot_lifts = np.array(boot_lifts)
    lift_ci_low, lift_ci_high = np.quantile(boot_lifts, [0.025, 0.975])
    boot_roi = (boot_lifts * 1300 - cost) / cost
    roi_ci_low, roi_ci_high = np.quantile(boot_roi, [0.025, 0.975])

    coef = model2.params[f"imp_{ch}T"]
    coef_ci = model2.conf_int().loc[f"imp_{ch}T"]

    rows.append({
        "channel": ch,
        "incremental_subscriptions": lift,
        "lift_ci95_low": lift_ci_low,
        "lift_ci95_high": lift_ci_high,
        "impressions": int(dfT[f"imp_{ch}"].sum()),
        "cost_usd": cost,
        "roi": roi,
        "roi_ci95_low": roi_ci_low,
        "roi_ci95_high": roi_ci_high,
        "interaction_coef_log_odds": coef,
        "interaction_coef_ci95_low": coef_ci[0],
        "interaction_coef_ci95_high": coef_ci[1],
        "interaction_p_value": model2.pvalues[f"imp_{ch}T"]
    })

roi_df = pd.DataFrame(rows).sort_values("roi", ascending=False)
display(roi_df)

Optimization terminated successfully.
         Current function value: 0.056043
         Iterations 11


,channel,incremental_subscriptions,lift_ci95_low,lift_ci95_high,impressions,cost_usd,roi,roi_ci95_low,roi_ci95_high,interaction_coef_log_odds,interaction_coef_ci95_low,interaction_coef_ci95_high,interaction_p_value
2,instagram,4.891805,4.533189,5.242890,5748,172.44,35.878605,33.175052,38.525383,0.051708,-0.137682,0.241099,5.925676e-01
1,twitter,721.945966,703.699999,736.337061,1727459,51823.77,17.110025,16.652324,17.471026,0.031590,0.022041,0.041140,8.962749e-11
4,youtube,-45.307340,-47.482239,-43.271597,331332,9939.96,-6.925531,-7.209976,-6.659286,-0.002351,-0.012901,0.008199,6.622892e-01
0,facebook,-123.024842,-129.533158,-114.873902,510864,15325.92,-11.435412,-11.987471,-10.744020,-0.007965,-0.023790,0.007860,3.239156e-01
3,letterboxd,-29.604503,-32.342272,-27.034738,60382,1811.46,-22.245765,-24.210534,-20.401565,-0.103728,-0.447284,0.239828,5.540119e-01


## 9) Decision Summary
Use the outputs above to guide channel allocation, re-test strategy, and confidence caveats tied to observed randomization imbalance and sparse bins.

In [14]:
df

,id,subscription,imp_facebook,imp_twitter,imp_instagram,imp_youtube,imp_letterboxd,treatment,total_impressions,impression_group,imp_facebookT,imp_twitterT,imp_instagramT,imp_letterboxdT,imp_youtubeT
0,917978073,1,0,0,0,0,0,1,0,0-40,0,0,0,0,0
1,630918244,1,0,0,0,0,0,1,0,0-40,0,0,0,0,0
2,663985446,1,0,0,0,0,0,1,0,0-40,0,0,0,0,0
3,572576568,1,0,0,0,0,0,1,0,0-40,0,0,0,0,0
4,673572395,1,0,23,0,0,0,1,23,0-40,0,23,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1200088,439485145,0,0,1,0,0,0,0,1,0-40,0,0,0,0,0
1200089,911803109,0,0,0,1,0,0,0,1,0-40,0,0,0,0,0
1200090,548212879,0,1,0,0,0,0,0,1,0-40,0,0,0,0,0
1200091,202665406,0,0,0,0,0,0,0,0,0-40,0,0,0,0,0
